# Does your model still rely on the same physics after the machine changes?

**A hands-on lesson in attribution stability, on the Open Density Limit Database.**

You train a disruption-precursor classifier on one set of Alcator C-Mod campaigns and deploy it on
a later set. You check the AUC. It is fine. Most people stop there.

This notebook asks the next question — *is it fine for the same reasons?* — and then asks the
question that most people get wrong even when they do ask the first one: **compared to what?**

Runtime: roughly 5–10 minutes on a laptop. Nothing here needs a cluster or a data agreement.

---

**Data.** Maris, A. D., Rea, C., Trevisan, G. L., & the Alcator C-Mod Team, *The Open Density Limit
Database*, MIT Plasma Science and Fusion Center (2025). CC BY. Download it, then set `DATA` below.

**Package.** `pip install disruptionpy-qfc`

In [ ]:
DATA = "DL_DataFrame.csv"  # <- path to the release

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from disruptionpy_qfc import compute_null_floor, compute_qfc, default_model
from disruptionpy_qfc.demo import build_demo_frames

source, target = build_demo_frames(DATA)
print(f"source: {len(source):5d} rows  {source['shot'].nunique():3d} shots  "
      f"positive rate {source['label'].mean():.3f}   (campaigns 2000, 2001, 2003)")
print(f"target: {len(target):5d} rows  {target['shot'].nunique():3d} shots  "
      f"positive rate {target['label'].mean():.3f}   (campaigns 2005, 2009)")

Each row is a 10 ms time point inside a discharge. The label marks whether that moment is inside the
precursor phase of the density limit. The 48 features are causal rolling-window statistics — mean,
spread, slope, autocorrelation and so on, computed over the **preceding** ten points — of six raw
signals: density, elongation, minor radius, plasma current, toroidal field, triangularity. Nothing
looks ahead, because a warning system cannot.

Notice already that the two environments differ in more than their inputs: the fraction of
precursor time points is not the same. Keep that in mind at the end.

## 1. The usual check: does it still predict?

In [ ]:
from sklearn.metrics import roc_auc_score

FEATURES = [c for c in source.columns if c not in ("shot", "time", "label")]
model = default_model(random_state=42).fit(source[FEATURES], source["label"])
auc = roc_auc_score(target["label"], model.predict_proba(target[FEATURES])[:, 1])
print(f"AUC on the later campaigns: {auc:.4f}")

That is a healthy number. By the usual standard this model transferred.

## 2. The question nobody asks: is it the same model, in the way that matters?

A model can keep its accuracy while completely rearranging what it leans on. If an operator reads
the top contributing signals to decide *which actuator to reach for*, that rearrangement matters as
much as the accuracy does.

So: take the deployed model, ask it what it relied on in each environment, and compare.

In [ ]:
result = compute_qfc(source, target, n_bootstrap=0)   # no CI yet; we want the picture first

src = np.array(result.source_importance)
tgt = np.array(result.target_importance)
order = np.argsort(src)[::-1][:12]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(order))
ax.bar(x - 0.2, src[order] / src.max(), 0.4, label="source (2000–2003)")
ax.bar(x + 0.2, tgt[order] / tgt.max(), 0.4, label="target (2005–2009)")
ax.set_xticks(x)
ax.set_xticklabels([result.feature_cols_used[i] for i in order], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("mean |attribution| (scaled)")
ax.set_title("What the deployed model leaned on, in each environment")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"QFC (rank agreement of the two vectors) = {result.qfc:.4f}")

A rank correlation around 0.94. The bars line up. It is very tempting to write in the paper that
*attribution structure is preserved across the campaign shift* and move on.

**Don't.** That number has no meaning yet, and here is why.

## 3. Compared to what? The no-shift floor

Split the **source campaigns alone** in half by discharge, and run the identical measurement between
the two halves. There is no shift there — same machine, same era, same everything. Whatever score
that produces is the score you get for free, from finite data and a stochastic learner.

Any real comparison has to beat that, not zero.

In [ ]:
floor = compute_null_floor(source, n_seeds=20)
print(f"no-shift floor: {floor.mean:.4f} ± {floor.sd:.4f}  over {floor.n_seeds} splits")
print(f"per-split values: {np.round(floor.per_seed, 3)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.axvspan(floor.mean - floor.sd * floor.exact_cut, floor.mean + floor.sd * floor.exact_cut,
           color="0.85", label="no shift, within the source")
ax.scatter(floor.per_seed, np.zeros(len(floor.per_seed)) + 0.35, marker="|", s=300, c="0.35")
ax.scatter([result.qfc], [0.35], marker="o", s=90, c="crimson", zorder=5,
           label=f"real shift, 2000–2003 → 2005–2009 ({result.qfc:.3f})")
ax.set_yticks([])
ax.set_xlabel("attribution rank agreement (QFC)")
ax.legend(frameon=False, loc="upper left", fontsize=8)
ax.set_title("The 0.94 is not impressive — the floor is just as high")
plt.tight_layout()
plt.show()

There it is. The no-shift floor sits right where the "real" result sits. The 0.94 was never evidence
of stability — it is roughly what this model, on this much data, scores against *itself*.

This is the single most important habit in the whole notebook: **a raw attribution correlation is
uninterpretable without its own no-shift floor.** The floor depends on the model, the number of
discharges, the number of features and the attribution method, so it has to be recomputed for every
setup — you cannot borrow someone else's threshold, including the one this package ships with.

## 4. Saying it properly

`screen()` does both steps and returns a verdict against the floor with the exact small-sample cut,
`t(0.975, n−1)·√(1+1/n)` — the Student-*t* interval for a single new observation, which is what an
observed QFC is.

In [ ]:
final = compute_qfc(source, target, null_floor=floor, n_bootstrap=100)
print(final.summary())

**INDISTINGUISHABLE** — and that is a genuine, reportable finding, not a failure of the experiment.
Across this particular era shift, on this machine, the model's reliance structure did not move more
than a no-shift split moves it.

## 5. What a real collapse looks like

So that *INDISTINGUISHABLE* does not read as "the metric never fires", here is the same measurement
on a different domain, from the study this package came out of — heart-sound recordings moving from a
hospital corpus to a community-clinic corpus, with four model classes:

| domain / shift | real QFC | no-shift floor | verdict |
|---|---|---|---|
| C-Mod, 2000–2003 → 2005–2009 (this notebook) | ≈ 0.94 | ≈ 0.95 | INDISTINGUISHABLE |
| Phonocardiogram, hospital → community clinic | 0.153 | 0.899 ± 0.024 | COLLAPSE |

Same procedure, same kind of number, opposite conclusion. In the second case the deployed model kept
a usable AUC on some conditions while relying on almost entirely different features — which is
exactly the failure this measurement exists to catch, and exactly what you cannot see by watching
accuracy alone.

## 6. What this does and does not tell you

Worth being strict about, because the honest scope is short:

- The dataset here is a **teaching subset** of the C-Mod catalogue, released for exactly this purpose.
  It deliberately contains some uninformative variables, and it is not the full research database.
- This is **one machine across an operational era**. The interesting question — whether a model
  carries its reliance structure from one *device* to another — is not tested here, and published
  cross-machine work suggests the most useful parameters differ markedly between tokamaks even when
  detection rates are similar.
- The source and target differ in their **label prior** as well as their inputs (≈ 0.51 vs ≈ 0.72
  positive rate). QFC measures how the fixed deployed model's reliance changes with the inputs; it is
  blind to a change in the labels alone. Detecting that needs target labels, which is the thing you
  do not have before deployment.
- A high QFC does **not** mean the model is right. It means it is consistent. Those are different
  properties and only one of them is testable before you have ground truth.

## Exercises

1. Re-run the floor with `n_seeds=5`. How much does the cut widen? Why does that make small-sample
   claims dangerous?
2. Swap `default_model()` for a gradient-boosted tree, or a shallower forest. Does the floor move?
   (It does. That is why the floor is not a universal constant.)
3. Drop `density` and its derived features and re-run. Does the verdict change — and should it?
4. Build a target that is *only* a covariate shift: scale one signal by 3× and re-screen. Then build
   one that only changes the labels. Predict which one QFC reacts to before you run it.